# 02 - File Handler
**Module:** `src/utils/file_handler.py`

## What it does?
Handles all file reading and writing for the generator:
 - saves generated lessons to  `data/generated`
 - loads hand-crafted example lesson from `data/examples`
 - manages the deduplication registry in `data/registry`

## What does this notebook explore?
1. Loading example lessons by grade band
2. Saving and loading a lesson manually
3. The deduplication registry -- how it grows and what it catches

In [1]:
import sys
sys.path.insert(0, '..')

from src.utils.file_handler import (
    load_example_by_grade,
    save_lesson,
    load_lesson,
    load_registry,
    combo_exists,
    register_combo,
    get_covered_skills,
)

import json

## Part 1: Loading Example Lessons by Grade Band
Our hand-crafted lessons live in `data/examples/`.
The file handler selects the right one by matching grade band.
These are used as few-shot examples in the prompt.

In [2]:
# Load and inspect the 9-12 example
example = load_example_by_grade("9-12")

print("Lesson ID     :", example["lesson_id"])
print("Grade Band    :", example["metadata"]["grade_band"])
print("Theme         :", example["metadata"]["theme"])
print("Primary Skill :", example["metadata"]["primary_skill"])
print("Lesson Type   :", example["metadata"]["lesson_type"])
print()
print("Hook preview  :", example["lesson_flow"]["hook"]["content"][:80], "...")

Lesson ID     : L-912-RDG-SPK-004
Grade Band    : 9-12
Theme         : History & Change
Primary Skill : Analyze an author's rhetorical choices and explain how they strengthen or weaken the argument
Lesson Type   : Text Explorer

Hook preview  : In 1963, a 34-year-old Baptist minister stood at the Lincoln Memorial in front o ...


In [3]:
example

{'lesson_id': 'L-912-RDG-SPK-004',
 'metadata': {'grade_band': '9-12',
  'ela_domain': 'Reading → Speaking',
  'lesson_type': 'Text Explorer',
  'theme': 'History & Change',
  'primary_skill': "Analyze an author's rhetorical choices and explain how they strengthen or weaken the argument",
  'voice_markers': ['Task Adherence', 'Prosody', 'Fluency & Fillers'],
  'estimated_duration_minutes': 8,
  'ccss_anchor': 'CCSS.ELA-Literacy.SL.11-12.1.c — Propel conversations by posing and responding to questions that probe reasoning and evidence; clarify, verify, or challenge ideas and conclusions.',
  'design_notes': 'Cross-domain lesson: reading analysis feeds spoken output. Historical text reduces cultural bias risk vs. contemporary politics. Three voice markers evaluated simultaneously — appropriate for 9-12 where full complexity is supported. No scaffolds — independent production expected at this band.'},
 'lesson_flow': {'hook': {'duration_seconds': 70,
   'content': "In 1963, a 34-year-old 

In [4]:
# Compare example lesson IDs across all grade bands
for band in ["K-2", "3-5", "6-8", "9-12"]:
    ex = load_example_by_grade(band)
    print(f"{band:5} → {ex['lesson_id']:20} | Theme: {ex['metadata']['theme']}")

K-2   → L-K2-SPK-001         | Theme: Nature & Animals
3-5   → L-35-SPK-005         | Theme: Adventure & Discovery
6-8   → L-68-SPK-003         | Theme: Ethics & Dilemmas
9-12  → L-912-RDG-SPK-004    | Theme: History & Change


In [5]:
# What happens with an invalid grade band?
try:
    load_example_by_grade("college")
except ValueError as e:
    print("Caught ValueError!")
    print(e)

Caught ValueError!
[file_handler] Invalid grade band 'college'. Must be one of: ['K-2', '3-5', '6-8', '9-12']


In [6]:
load_example_by_grade("college")

ValueError: [file_handler] Invalid grade band 'college'. Must be one of: ['K-2', '3-5', '6-8', '9-12']

## Part 2: Saving and Loading a Lesson
Let's manually create a minimal lesson dict,
save it to `data/generated/`, and load it back.
This is exactly what the generator does after every successful generation.

In [7]:
# Create a minimal lesson dict (simulating generator output)
test_lesson = {
    "lesson_id": "L-NB-TEST-001",
    "metadata": {
        "grade_band": "6-8",
        "ela_domain": "Speaking",
        "lesson_type": "Debate Drop",
        "theme": "Notebook Test",
        "primary_skill": "Present a claim with supporting evidence",
        "voice_markers": ["Prosody", "Task Adherence"],
        "estimated_duration_minutes": 7,
        "ccss_anchor": "CCSS.ELA-Literacy.SL.6.4"
    },
    "lesson_flow": {
        "hook": {"duration_seconds": 70, "content": "A town must decide..."},
        "model": {"duration_seconds": 90, "content": "Here is a strong argument...",
                  "skill_named_explicitly": "Today we are practicing: making a claim."},
        "practice": [
            {"prompt_id": "P1", "type": "supported",
             "text": "State your position.", "scaffold": "I believe that..."}
        ],
        "reflect": {"duration_seconds": 50, "voice_marker_focus": "Prosody",
                    "positive_signal": "Strong emphasis on key claims.",
                    "growth_signal": "Slow down on your evidence."}
    },
    "guardrail_flags": {
        "single_skill_check":       {"status": "pass", "message": "One skill confirmed"},
        "vocabulary_ceiling_check": {"status": "pass", "message": "Within ceiling"},
        "cognitive_load_check":     {"status": "pass", "message": "Appropriate for 6-8"},
        "cultural_bias_check":      {"status": "pass", "message": "No bias detected"}
    }
}

filepath = save_lesson(test_lesson)


[file_handler] Saved lesson → D:\projects\bantrly-lesson-generator\data\generated\L-NB-TEST-001.json


In [8]:
# Load it back and verify it's identical
loaded = load_lesson("L-NB-TEST-001")

print("Loaded lesson_id :", loaded["lesson_id"])
print("Grade band       :", loaded["metadata"]["grade_band"])
print("Primary skill    :", loaded["metadata"]["primary_skill"])
print()

# Confirm it round-trips perfectly
assert loaded == test_lesson, "Round-trip failed!"
print("Round-trip check passed — saved and loaded lesson are identical")

Loaded lesson_id : L-NB-TEST-001
Grade band       : 6-8
Primary skill    : Present a claim with supporting evidence

Round-trip check passed — saved and loaded lesson are identical


In [9]:
# What happens if we try to load a lesson that doesn't exist?
try:
    load_lesson("akhsdbahksd")
except FileNotFoundError as e:
    print("Caught FileNotFoundError!")
    print(e)

Caught FileNotFoundError!
[file_handler] No lesson found with ID 'akhsdbahksd' at D:\projects\bantrly-lesson-generator\data\generated\akhsdbahksd.json


## Part 3: The Deduplication Registry
Every generated lesson logs its `theme × skill × grade_band` combination
to `data/registry/registry.json`. Before generating, we check this log
to avoid producing the same lesson twice.

This directly addresses the brief requirement: *"Avoid repeating content."*

Research grounding: spaced repetition (Ebbinghaus, 1885) shows that
varied practice across different contexts produces better retention than
repeating the same task. Our dedup system enforces this at the content level.

In [10]:
# Check the current state of the registry
registry = load_registry()
registry

{'used_combinations': [{'theme': 'politics',
   'skill': 'Explain a process step by step',
   'grade_band': '3-5',
   'lesson_id': 'L-NB-TEST-002',
   'generated_at': '2026-03-12T17:34:02.094052'},
  {'theme': 'Ocean Ecosystems',
   'skill': 'Explain a process step by step',
   'grade_band': '3-5',
   'lesson_id': 'L-NB-TEST-002',
   'generated_at': '2026-03-12T17:34:15.994680'},
  {'theme': 'Ocean Ecosystems',
   'skill': 'Describe a habitat using sensory details and basic needs',
   'grade_band': '3-5',
   'lesson_id': 'L-35-SPK-009',
   'generated_at': '2026-03-12T19:54:01.502037'},
  {'theme': 'Climate Change',
   'skill': "Analyze an author's use of evidence to support a claim about a scientific issue",
   'grade_band': '9-12',
   'lesson_id': 'L-912-RDG-SPK-009',
   'generated_at': '2026-03-12T19:55:22.297580'},
  {'theme': 'Persuasive Writing',
   'skill': 'Analyze and explain the effectiveness of persuasive techniques in a written text',
   'grade_band': '9-12',
   'lesson_id':

In [11]:
combos = registry["used_combinations"]

print(f"Registry contains {len(combos)} combination(s):\n")
for entry in combos:
    print(f"  [{entry['grade_band']}] x {entry['theme']} × {entry['skill'][:50]}")
    print(f"         lesson_id: {entry['lesson_id']} | generated: {entry.get('generated_at', 'N/A')}")
    print()

Registry contains 20 combination(s):

  [3-5] x politics × Explain a process step by step
         lesson_id: L-NB-TEST-002 | generated: 2026-03-12T17:34:02.094052

  [3-5] x Ocean Ecosystems × Explain a process step by step
         lesson_id: L-NB-TEST-002 | generated: 2026-03-12T17:34:15.994680

  [3-5] x Ocean Ecosystems × Describe a habitat using sensory details and basic
         lesson_id: L-35-SPK-009 | generated: 2026-03-12T19:54:01.502037

  [9-12] x Climate Change × Analyze an author's use of evidence to support a c
         lesson_id: L-912-RDG-SPK-009 | generated: 2026-03-12T19:55:22.297580

  [9-12] x Persuasive Writing × Analyze and explain the effectiveness of persuasiv
         lesson_id: L-912-RDG-SPK-009 | generated: 2026-03-12T19:55:53.660668

  [9-12] x Persuasive Writing × Analyze and explain the effectiveness of persuasiv
         lesson_id: L-912-RDG-SPK-009 | generated: 2026-03-12T20:15:33.662847

  [9-12] x Persuasive Writing × Analyze the use of pathos, ethos

In [15]:
# Demonstrate combo_exists before and after registration
theme      = "Ocean Ecosystems"
skill      = "Explain a process step by step"
grade_band = "3-5"
ela_domain = "Speaking"

print("Before registration:")
print(f"  combo_exists(...) → {combo_exists(theme, skill, grade_band, ela_domain)}")
print()
# Register it
register_combo(theme, skill, grade_band, ela_domain, "L-NB-TEST-002")

print("\nAfter registration:")
print(f"  combo_exists(...) → {combo_exists(theme, skill, grade_band, ela_domain)}")


Before registration:
  combo_exists(...) → False

[file_handler] Registered combo → 3-5 | Ocean Ecosystems | Explain a process step by step

After registration:
  combo_exists(...) → True


In [17]:
# Case-insensitive check
print("Case-insensitive check:")
print(f"  lowercase theme  → {combo_exists('ocean ecosystems', skill, grade_band, ela_domain)}")
print(f"  uppercase theme  → {combo_exists('OCEAN ECOSYSTEMS', skill, grade_band, ela_domain)}")

Case-insensitive check:
  lowercase theme  → True
  uppercase theme  → True


In [18]:
# View the full registry now
registry = load_registry()
print("Full registry contents:\n")
print(json.dumps(registry, indent=2))

Full registry contents:

{
  "used_combinations": [
    {
      "theme": "politics",
      "skill": "Explain a process step by step",
      "grade_band": "3-5",
      "lesson_id": "L-NB-TEST-002",
      "generated_at": "2026-03-12T17:34:02.094052"
    },
    {
      "theme": "Ocean Ecosystems",
      "skill": "Explain a process step by step",
      "grade_band": "3-5",
      "lesson_id": "L-NB-TEST-002",
      "generated_at": "2026-03-12T17:34:15.994680"
    },
    {
      "theme": "Ocean Ecosystems",
      "skill": "Describe a habitat using sensory details and basic needs",
      "grade_band": "3-5",
      "lesson_id": "L-35-SPK-009",
      "generated_at": "2026-03-12T19:54:01.502037"
    },
    {
      "theme": "Climate Change",
      "skill": "Analyze an author's use of evidence to support a claim about a scientific issue",
      "grade_band": "9-12",
      "lesson_id": "L-912-RDG-SPK-009",
      "generated_at": "2026-03-12T19:55:22.297580"
    },
    {
      "theme": "Persuasive Wr

## Part 4: Skill Coverage Tracking
`get_covered_skills()` reads the registry and returns all skills
already generated for a given grade band + domain combination.
This is what `skill_selector.py` calls to know which skills
have been covered and which are still remaining.

In [25]:
# See which skills have been covered for 3-5 Speaking
covered = get_covered_skills("3-5", "Listening")

print(f"Covered skills for 3-5 Speaking ({len(covered)}/5 covered):\n")
for i, skill in enumerate(covered, 1):
    print(f"  {i}. {skill}")

if not covered:
    print("  None yet — generate some lessons first.")

Covered skills for 3-5 Speaking (0/5 covered):

  None yet — generate some lessons first.


In [26]:
# Compare coverage across domains for the same grade band
print("Coverage snapshot for grade band 3-5:\n")
for domain in ["Speaking", "Listening", "Reading", "Writing"]:
    covered = get_covered_skills("3-5", domain)
    print(f"  {domain:<20} : {len(covered)} skill(s) covered")

Coverage snapshot for grade band 3-5:

  Speaking             : 3 skill(s) covered
  Listening            : 0 skill(s) covered
  Reading              : 2 skill(s) covered
  Writing              : 1 skill(s) covered


## Summary

| What we explored | Key takeaway |
|---|---|
| `load_example_by_grade()` | Grade band determines which few-shot example the LLM sees |
| `save_lesson()` / `load_lesson()` | Lessons persist to disk as JSON, round-trip perfectly |
| Missing lesson raises `FileNotFoundError` | Errors are descriptive, not cryptic |
| `combo_exists()` | Registry correctly tracks used combinations |
| Case-insensitive matching | "Ocean" and "ocean" are treated as the same theme |
| `register_combo()` | Now takes `ela_domain` as a required argument — registry entries track domain for accurate coverage filtering |
| `get_covered_skills()` | New function — returns skills already covered for a band + domain; used by `skill_selector.py` |
**Design decision this validates:**
The registry is append-only and human-readable JSON. It's not a database.
This enforces the deduplication requirement from the brief.

**Known limitation:**
The registry tracks `theme × skill × grade_band`. Because the LLM picks
the skill, two requests with the same theme could produce different skills
and both pass the dedup check. A production system would use semantic
similarity (embeddings) to catch near-duplicate lessons.